In [ ]:
import rospy
import actionlib
from geometry_msgs.msg import PoseStamped
from IPython.display import display
from ipywidgets import IntText, Button, HBox, VBox, Label

import assignment_2_2024.msg

In [ ]:
# Initializing the ROS node
if not rospy.core.is_initialized():
    rospy.init_node('jupyter_client_node', anonymous=True)

# Creating the action client
client = actionlib.SimpleActionClient('reaching_goal', assignment_2_2024.msg.PlanningAction)
client.wait_for_server()

current_feedback = None

In [ ]:
# Function to update feedback
def update_feedback(fd): 
    global current_feedback
    current_feedback = fd
    
# Function to set and send the goal
def send_goal(b):
    x = x_input.value
    y = y_input.value

    if(x < 10.0 and x > -10.0 and y < 10.0 and y > -10.0):
        goal = assignment_2_2024.msg.PlanningGoal()
        goal.target_pose = PoseStamped()

        goal.target_pose.pose.position.x = x
        goal.target_pose.pose.position.y = y
                
        client.send_goal(goal, feedback_cb = update_feedback)

        rospy.loginfo(f"Goal sent: ({x}, {y})")
        status_label.value = f"Goal sent: ({x}, {y})"
        
    else:
        rospy.loginfo("Target out of boundaries")        
        status_label.value = "Target out of boundaries"


# Function to delete the goal
def cancel_goal(b):
    client.cancel_goal()
    rospy.loginfo("Goal cancelled")
    status_label.value = "Goal cancelled"

In [ ]:
# Function the cancel button activate 
def cancel_goal_btn(b):
    if client.get_state() not in [actionlib.GoalStatus.SUCCEEDED, actionlib.GoalStatus.ABORTED, actionlib.GoalStatus.PREEMPTED]:
        rospy.loginfo("Cancelling the current goal.")
        client.cancel_goal()
    else:
        rospy.loginfo("The goal has already been achieved.")

# Function the feedback button activate 
def update_feedback_btn(b):
    rospy.loginfo("Requesting feedback...\n")
    if current_feedback is None:
        rospy.loginfo("No feedback has been recieved")
        status_label.value = "No feedback has been recieved"
    else:
        rospy.loginfo(f"Latest feedback:\n {current_feedback}")
        status_label.value = f"Latest feedback:\n {current_feedback}"


In [ ]:
# UI widgets
x_input = IntText(value = 0, description =  'X: ')
y_input = IntText(value = 0, description =  'Y: ')

send_btn = Button(description = "Send Goal", button_style = 'success')
cancel_btn = Button(description = "Cancel Goal", button_style = 'danger')
feedback_btn = Button(description = "FeedBack")

status_label = Label(value = "")

send_btn.on_click(send_goal)
cancel_btn.on_click(cancel_goal_btn)
feedback_btn.on_click(update_feedback_btn)

# Layout
ui = VBox ([
    HBox([x_input, y_input]),
    HBox([send_btn, cancel_btn]),
    HBox([feedback_btn]),
    status_label
])

display(ui)